# Lab 2a: The minimal agent, line by line

Kiel University · Agentic AI (infAgAI-01a) · Winter 2026

Companion to the lecture's code walkthrough *mini_agent.py, line by line*. In about fifty lines of Python, this notebook builds the agent that the whole course grows from, one part at a time, and lets you watch every line run. It is written for readers who are new to Python, new to agents, or both.

**Learning objectives.** After this lab you can:

- read and explain every line of the minimal agent: setup and tool, model call, tool execution, loop,
- describe a tool to a model with a **JSON schema** and connect the model's tool name to a Python function through a **dispatch table**,
- explain why the model **never runs code**, and why the **whole message list** is sent every round,
- turn every tool failure into an **observation** instead of a crash,
- name the loop's **exits** (final answer, cut-off answer, step cap), read Python's `for ... else`, and run the finished agent on a question of your own,
- use the Python features the agent relies on: lists, dictionaries, `try`/`except`, `fn(**args)`, f-strings and slicing.

*Estimated effort: about 60 minutes. This notebook is not graded and has no hand-in: every gap (marked with underscores) and every question has a fold-out solution. Try first, then look. It makes about ten requests to the model, so your Academic Cloud key from the Onboarding notebook must be set up.*


## Overview: four parts, one loop

The program has four parts. Read top to bottom, they are also the order in which things happen: part 1 sets things up, parts 2 and 3 define what one round does, and part 4 is the loop that repeats those two things until the run ends.


<div style="text-align: center;">
  <img src="images/fig01_program_map.png" width="960">
</div>
<p style="text-align: center; color: #475569; font-size: 0.92em;"><i>The lecture's file (left) and what runs when (right). The colours mark the four parts throughout this lab; where the lecture calls call_model, this lab calls llm.chat.</i></p>


## Theory recap: an agent is a loop around a model

**The working definition.** An LLM agent is a system in which a language model decides, in a loop, which actions to take next on the way to a goal, calling tools and observing their results. The model itself is **stateless**: each request starts from zero, and it only ever returns text. Everything that makes it an agent lives in a small program around it, the **runtime**.

**One round of the loop** has four phases:

1. **Reason.** The runtime sends the *entire* conversation so far, the **message list**, to the model. The model answers with text or with one or more **tool calls**.
2. **Act.** The runtime, not the model, runs the requested Python function. A tool call is only a request written as text.
3. **Observe.** The result is appended to the message list as a new message. The model sees it in the next round, because the whole list is sent again.
4. **Decide.** There is no separate decision inside the model: the *shape* of the answer is the signal. A tool call means "go round again"; no tool call means the run ends (completed, or refused, or cut off: the exit is **classified**, not assumed).

**The message list is the agent's entire working state.** Nothing else carries over from one round to the next.

**Stop conditions.** A loop around a model needs exits that do not depend on the model behaving well. The minimal agent has two: the model requests no further action, and a **step cap** `MAX_STEPS` that ends the run after a fixed number of rounds, whatever the model does.

**Errors become observations.** An invented tool name, malformed arguments or a failing tool must not crash the agent. The runtime turns each failure into a short `TOOL ERROR` text, appends it like any other result, and lets the model react in the next round.


## Part A: Setup: who talks to whom

An agent program talks to a model that runs somewhere else, on a **provider's** server. The **client** is the part of our program that sends the requests and turns the answers into Python data. In the lecture the client is the `openai` library; in the labs it is our small file `llm.py`, which speaks the same *Chat Completions* format that the Academic Cloud (GWDG) accepts. One difference to keep in mind: the lecture's client returns objects that you read with dots (`msg.content`), ours returns plain dictionaries that you read with square brackets (`msg["content"]`). Everything else carries over line by line.


<div style="text-align: center;">
  <img src="images/fig02_client_provider.png" width="960">
</div>


In [ ]:
import json          # json: tool arguments arrive as JSON text, json.loads turns them into a dictionary

import llm           # llm.py, the course client: llm.chat(messages, ...) sends one request

MODEL = llm.MODEL    # which model answers; it must be able to return tool calls, not only text

try:
    resp = llm.chat([{"role": "user", "content": "Reply with exactly one word: ready"}], model=MODEL)
    print(f"Model '{MODEL}' answered:",
          resp["choices"][0]["message"]["content"].strip())
except Exception as e:
    print("Could not talk to the model:", e)
    print("Run the Onboarding notebook first, then re-run this cell.")


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `import json` brings in a module from Python's standard library. JSON is the text format in which programs (and models) exchange data; we need `json.loads` in part 3 to turn the model's argument text into a dictionary.
- `import llm` loads the file `llm.py` that lies next to this notebook. It contains `llm.chat()`, the function that sends a request to the Academic Cloud, and `llm.MODEL`, the name of the course model. The key is read from the file that the Onboarding notebook saved; it never appears in the notebook.
- `MODEL = llm.MODEL` copies the model name into our own variable, so that every call can say `model=MODEL`, exactly as in the lecture's code (`MODEL = "your-model"`).
- The `try` block sends one tiny message. If the call succeeds, the reply is printed. If anything goes wrong (no key, no network, overloaded server), Python jumps to `except Exception as e:` instead of stopping; `e` holds the error, which we print together with a hint.
- `resp["choices"][0]["message"]["content"]` is the path to the reply text inside the response, read from the outside in: the list of choices, its first entry, that entry's message, the message's text. Part D takes this apart.

</details>


> **Compare with the lecture.** The lecture writes `client = OpenAI()` and then `client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)`. Our `llm.chat(messages, model=MODEL, tools=TOOLS)` sends the identical request; `llm.py` adds the server address and your key. In both cases **no secret appears in the code**.

> **Key point.** `MODEL` is just a string sent with every request. It must name a model that can return **tool calls**, not only text; otherwise the loop in part 4 never gets an action to execute.


❓ **Q:** What does "OpenAI-compatible" mean for a provider like the Academic Cloud? *(not exam-relevant)*
<details><summary>Click for answer</summary>

It accepts requests in the same format as OpenAI's Chat Completions API (the same JSON fields: `model`, `messages`, `tools`, ...) and answers in the same format. Therefore the same client code works for many providers; only the server address, the key and the model names differ. That is also why the lecture's program carries over to our client almost word for word.

</details>


## Part B: Python warm-up: lists and dictionaries

Agent code is full of two data shapes, because JSON, the format in which programs and models exchange data, has exactly these two:

- a **list** `[a, b, c]`: items in order, accessed by **position**, counted from 0;
- a **dictionary** `{"key": value}`: a lookup table from **key** to value.

Fill the gaps, run, and check each printed value against the comment.


In [ ]:
fruits = ["apple", "pear", "plum"]
print(fruits[___])          # -> apple   (the first item: Python counts from 0)
print(len(fruits))           # -> 3
fruits.append("fig")         # add a fourth item at the end
print(fruits)                # -> ['apple', 'pear', 'plum', 'fig']

person = {"name": "Anna", "age": 30}
print(person[___])       # -> Anna    (look up by key)
person["city"] = "Kiel"      # assigning to a new key adds it
print(person)                # -> {'name': 'Anna', 'age': 30, 'city': 'Kiel'}


<details>
<summary><b>Click here for the solution</b></summary>

```python
fruits = ["apple", "pear", "plum"]
print(fruits[0])          # -> apple   (the first item: Python counts from 0)
print(len(fruits))           # -> 3
fruits.append("fig")         # add a fourth item at the end
print(fruits)                # -> ['apple', 'pear', 'plum', 'fig']

person = {"name": "Anna", "age": 30}
print(person["name"])       # -> Anna    (look up by key)
person["city"] = "Kiel"      # assigning to a new key adds it
print(person)                # -> {'name': 'Anna', 'age': 30, 'city': 'Kiel'}
```

</details>


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `fruits = ["apple", "pear", "plum"]` creates a list of three strings. `fruits[0]` is the first item: positions are counted from 0, so the last of three items is `fruits[2]`.
- `len(fruits)` is the number of items. `fruits.append("fig")` adds an item at the end; the list now has four items, and `fruits[3]` would be `"fig"`.
- `person = {"name": "Anna", "age": 30}` creates a dictionary with two keys. `person["name"]` looks a value up by its key, like looking a word up in a real dictionary. Keys are usually strings and must be written with quotes.
- `person["city"] = "Kiel"` assigns a value to a key that did not exist yet; this adds the pair to the dictionary. Assigning to an existing key would replace its value.
- `print(person)` shows the whole dictionary. The message list of the agent and every message in it are built from exactly these two shapes.

</details>


Values can themselves be lists or dictionaries: the shapes **nest**. Reading rule: work from the **outside in**. Each `[ ]` opens one level, and you pick one step per level: a position for a list, a key for a dictionary.


In [ ]:
person = {"name": "Anna", "pets": ["cat", "dog"]}      # a list inside a dictionary
team = [{"name": "Anna"}, {"name": "Ben"}]             # dictionaries inside a list

print(person["pets"][___])     # -> dog   key first, then position
print(team[___]["name"])       # -> Ben   position first, then key

msg = {"role": "user", "content": "Write a short report on the EU AI Act."}
print(msg["role"])           # -> user  (a message, as in part 2, is a dictionary too)


<details>
<summary><b>Click here for the solution</b></summary>

```python
person = {"name": "Anna", "pets": ["cat", "dog"]}      # a list inside a dictionary
team = [{"name": "Anna"}, {"name": "Ben"}]             # dictionaries inside a list

print(person["pets"][1])     # -> dog   key first, then position
print(team[1]["name"])       # -> Ben   position first, then key

msg = {"role": "user", "content": "Write a short report on the EU AI Act."}
print(msg["role"])           # -> user  (a message, as in part 2, is a dictionary too)
```

</details>


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `person["pets"]` is the list `["cat", "dog"]`; adding `[1]` picks its second item. Two bracket pairs, two levels: first a key (dictionary), then a position (list).
- `team[1]` is the second dictionary in the list; adding `["name"]` looks up its name. Same idea, other order: first a position, then a key.
- `msg` is a **message** in the shape the model expects: a dictionary with the keys `role` (who is speaking) and `content` (the text). The agent's message list is a list of such dictionaries, so `messages[0]["role"]` is the role of the first message.

</details>


❓ **Q:** What happens if you run `person["city"]` *before* the key `"city"` was added?
<details><summary>Click for answer</summary>

Python raises a `KeyError`: the key does not exist. Part 3 of the agent relies on exactly this behaviour: looking up a tool name that is not in the dispatch table raises `KeyError`, which the agent catches and turns into an observation.

</details>


## Part C: Part 1 of the agent: the tool and its description

### The tool is an ordinary function

A **tool** is a plain Python function. Ours searches a tiny offline "web" of three made-up pages, so that everything runs without internet access. *(The facts are simplified teaching material.)* The cell is given; read it and the explanation, then run it.


In [ ]:
PAGES = {   # page_id -> (title, text)
    "eu_ai_act": ("EU AI Act overview",
                  "The EU AI Act is a regulation that sets risk-based rules for AI systems placed on "
                  "the EU market. It entered into force on 1 August 2024; most obligations apply "
                  "from 2 August 2026."),
    "risk_levels": ("Four risk levels",
                    "The Act sorts AI systems into four levels: unacceptable risk (banned), high risk, "
                    "limited risk (transparency duties) and minimal risk."),
    "penalties": ("Penalties",
                  "Fines for prohibited practices reach up to 35 million euro or 7 percent of "
                  "worldwide annual turnover."),
}

def web_search(query: str) -> str:
    '''Search the offline web, return the top hits as text.'''
    words = query.lower().split()
    scored = []
    for page_id, (title, text) in PAGES.items():        # key and value, pair by pair
        score = sum(w in (title + " " + text).lower() for w in words)
        if score > 0:
            scored.append((score, page_id, title, text))
    scored.sort(reverse=True)                            # best match first
    hits = []
    for score, page_id, title, text in scored[:2]:       # the two best pages
        hits.append(f"[{page_id}] {title}: {text}")
    if not hits:
        return "no results"
    return "\n".join(hits)

print(web_search("EU AI Act"))


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `PAGES` is a dictionary from a page id to a pair `(title, text)`. A pair in round brackets is a **tuple**: like a list, but it cannot be changed afterwards. The long texts are written as several string pieces one below the other; Python joins adjacent string literals into one string.
- `def web_search(query: str) -> str:` defines a function with one argument. The **type hints** `: str` and `-> str` say "takes a string, returns a string"; they document the intent, Python does not enforce them. The text in triple quotes below the `def` line is the **docstring**, a description for human readers.
- `query.lower().split()` lower-cases the query and splits it at spaces into a list of words.
- `for page_id, (title, text) in PAGES.items():` goes through the dictionary pair by pair. `items()` hands out `(key, value)` pairs; here the value is itself a pair, the tuple `(title, text)`. The brackets in the `for` line take that pair apart on the spot: `page_id` receives the key, `title` and `text` receive the two parts of the value. Without this shorthand you would write `for page_id, page in PAGES.items():` and then `title = page[0]` and `text = page[1]`.
- `sum(w in (title + " " + text).lower() for w in words)` is a compact loop: `for w in words` takes the query words one at a time and calls the current one `w`; `w in ...` asks whether that word occurs in the page text, giving `True` or `False`; `sum` adds these up, and because Python counts `True` as 1, the result is the number of matching words. That is the page's score.
- Pages with a score above 0 are collected as tuples `(score, page_id, title, text)`. `scored.sort(reverse=True)` sorts these tuples; tuples are compared by their first element, so the highest score comes first.
- `scored[:2]` is a **slice**: the first two items of the sorted list. The loop takes each of these tuples apart into four names (`score` is not used afterwards, but the `for` line has to name all four parts) and builds one text line per hit with an f-string: the page id in square brackets, the title, the text.
- `if not hits:` catches the case that no page matched; the function then returns the fixed text `"no results"`, which the model will read as its observation. Otherwise `"\n".join(hits)` turns the list of hit strings into one string: `join` is called on the separator, here `"\n"`, which is not a backslash and an n but Python's way of writing the single newline character; it puts that separator between every two entries of the list. With two hits the result is one string that prints as two lines, one per page.

</details>


> **Key point.** The model never sees any of these lines: not the name of the Python function, not the docstring, not the body. It only sees the description we write next.


### Describing the tool to the model: `TOOLS`

Because the model cannot see the function, we describe it in JSON: a name, a sentence in plain English, and the arguments with their types. `TOOLS` is a list (an agent usually has several tools) of dictionaries, nested five levels deep. Fill the gaps, then walk the levels in the two `print` lines.


<div style="text-align: center;">
  <img src="images/fig03_tools_nesting.png" width="960">
</div>


In [ ]:
TOOLS = [{"type": "function", "function": {
    "name": ___,
    "description": "Search the web for a query.",
    "parameters": {"type": "object",
        "properties": {"query": {"type": "string"}},
        "required": [___]}}}]

# Walk the levels from the outside in:
print(TOOLS[0]["function"][___])                                     # -> web_search
print(TOOLS[0]["function"]["parameters"]["properties"]["query"]["type"])   # -> string


<details>
<summary><b>Click here for the solution</b></summary>

```python
TOOLS = [{"type": "function", "function": {
    "name": "web_search",
    "description": "Search the web for a query.",
    "parameters": {"type": "object",
        "properties": {"query": {"type": "string"}},
        "required": ["query"]}}}]

# Walk the levels from the outside in:
print(TOOLS[0]["function"]["name"])                                     # -> web_search
print(TOOLS[0]["function"]["parameters"]["properties"]["query"]["type"])   # -> string
```

</details>


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `TOOLS` is a list with one dictionary, because this agent has one tool. `"type": "function"` says: this tool is a function of ours, and our runtime will run it.
- `"name"` is the name the model will write into its tool calls. It must be spelled exactly like the key in `FUNCS` (next cell); otherwise the runtime cannot find the function.
- `"description"` is read by the model like part of the prompt. A vague or misleading description leads to wrong tool calls; writing it is prompt engineering.
- `"parameters"` is a **JSON Schema**, a standard way of describing data: `"type": "object"` (the arguments form a dictionary), `"properties"` (one entry per argument, with its type; here one string argument `query`), `"required"` (the arguments that must always be present).
- The `print` lines read the structure from the outside in, one bracket pair per level: position `[0]` in the list, then the keys `"function"`, `"name"` (or `"parameters"`, `"properties"`, `"query"`, `"type"`). This is the reading rule from Part B applied to real agent data.

</details>


> **Key point.** This JSON is everything the model knows about the tool. It is a contract about **shape**, not a guarantee about sense.

### What a schema is: a blank form

A schema fixes the fields, their types and which are mandatory. The model's arguments are the **filled-in form**. The form says nothing about whether the entries make sense: `{"query": "EU AI Act"}` fits, `{"query": 42}` has the wrong type, `{}` lacks the required field, `{"q": "EU AI Act"}` uses a wrong name. Our program never checks this itself; the provider usually does, and part 3 catches whatever slips through.


<div style="text-align: center;">
  <img src="images/fig04_schema_form.png" width="960">
</div>


### The dispatch table `FUNCS`

When the model answers with the name `"web_search"`, the runtime needs the real function. `FUNCS` is a dictionary from **tool name** (text) to **Python function**: the only bridge between the two worlds. Fill the gap.


<div style="text-align: center;">
  <img src="images/fig05_name_bridge.png" width="960">
</div>


In [ ]:
FUNCS = {"web_search": ___}    # name for the model -> Python function (no quotes: the function itself)

fn = FUNCS["web_search"]                # look the name up ...
print(fn("risk levels")[:70])           # ... and call what we found
print("search_web" in FUNCS)            # an invented name -> False: there is no path to any code


<details>
<summary><b>Click here for the solution</b></summary>

```python
FUNCS = {"web_search": web_search}    # name for the model -> Python function (no quotes: the function itself)

fn = FUNCS["web_search"]                # look the name up ...
print(fn("risk levels")[:70])           # ... and call what we found
print("search_web" in FUNCS)            # an invented name -> False: there is no path to any code
```

</details>


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- In `{"web_search": web_search}` the key is a **string** (with quotes), the value is the **function itself** (no quotes, no parentheses). Writing `web_search()` with parentheses would call the function right here and store its result instead.
- `fn = FUNCS["web_search"]` looks the function up and gives it a second name, `fn`. Functions are values in Python: they can be stored in dictionaries and passed around like numbers or strings.
- `fn("risk levels")` calls the function through that second name; `[:70]` shows only the first 70 characters of the result.
- `"search_web" in FUNCS` asks whether a key exists in the dictionary. For an invented name the answer is `False`: nothing that is not in this table can ever be executed, however convincingly the model asks for it.

</details>


> **Common mistake.** Spelling the name differently in `TOOLS` and in `FUNCS`. The model uses the name from `TOOLS`; if `FUNCS` has another spelling, every call fails with *unknown tool*.


❓ **Q:** If the model cannot see the function, how does it decide when to call it and with which arguments?
<details><summary>Click for answer</summary>

Only from the JSON description: the name, the plain-English description and the parameter schema. That makes writing tool descriptions a form of prompt design: the description must say what the tool does, when it helps and what it returns, including its limits.

</details>


❓ **Q:** Why is the dispatch table a safety feature and not just a convenience?
<details><summary>Click for answer</summary>

Anything that is not in `FUNCS` cannot be executed, however convincingly the model asks for it. An invented or manipulated name leads to a `KeyError`, which the agent turns into an error message, never to running code that was not registered on purpose.

</details>


## Part D: Part 2 of the agent: the model call

### The conversation starts here

Before the loop begins, the message list holds two dictionaries: the **system prompt** (standing instructions from the developer, sent in every round) and the **task** (what the user wants done). Fill the two roles.


In [ ]:
SYSTEM_PROMPT = ("You are a careful research assistant. Use web_search to find facts. "
                 "Answer in two or three sentences and cite page ids in square brackets.")
task = "Write a short report on the EU AI Act."

messages = [
    {"role": ___, "content": SYSTEM_PROMPT},
    {"role": ___, "content": task},
]
print(len(messages), "messages:", [m["role"] for m in messages])


<details>
<summary><b>Click here for the solution</b></summary>

```python
SYSTEM_PROMPT = ("You are a careful research assistant. Use web_search to find facts. "
                 "Answer in two or three sentences and cite page ids in square brackets.")
task = "Write a short report on the EU AI Act."

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": task},
]
print(len(messages), "messages:", [m["role"] for m in messages])
```

</details>


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `SYSTEM_PROMPT` is one string written as two pieces inside parentheses. It tells the model its role, which tool exists, and how to answer. The model treats the `system` role as instructions from the operator and reads it again in every round.
- `task` is the user's request. In the lecture both are "defined earlier"; here they are.
- `messages` is a list of two message dictionaries, oldest first. This list is the agent's working state: every answer and every tool result will be appended to it.
- `[m["role"] for m in messages]` is a list comprehension: for each message `m`, take its role. It prints `['system', 'user']`.

</details>


### One request, and the bounded retries you do not see

The model call itself is one line: `llm.chat(messages, model=MODEL, tools=TOOLS)`. The lecture wraps that line in a function `call_model` with **bounded retries**, because the network is unreliable: timeouts and short outages happen and usually pass by themselves, so the request is tried up to three times with growing pauses (1, 2, 4 seconds) and then fails loudly. Our `llm.py` does exactly this inside `llm.chat`, so the labs call it directly:

```python
def call_model(messages, retries=3):            # the lecture's version
    for attempt in range(retries):              # attempt = 0, 1, 2
        try:
            return client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)
        except Exception as e:                  # network or API error
            print("API error:", e)
            time.sleep(2 ** attempt)            # wait 1, 2, 4 seconds, then retry
    raise RuntimeError("model unavailable after retries")
```

> **Common mistake.** `while True:` try again. Against a dead server this hangs forever. Bounded retries fail loudly instead.

### What comes back: anatomy of one answer

Now one real request. The response is nested dictionaries and lists; you reach its parts with keys and positions. Fill the gaps, then compare the printout with the figure.


<div style="text-align: center;">
  <img src="images/fig06_response_anatomy.png" width="960">
</div>


In [ ]:
resp = llm.chat(messages, model=MODEL, tools=TOOLS)
msg = resp["choices"][0][___]

print("finish_reason:", resp["choices"][0]["finish_reason"])
print("content:      ", msg["content"])

if not msg.get("tool_calls"):
    print("The model answered without a tool call this time. Run the cell again.")
else:
    call = msg["tool_calls"][___]               # the first (usually: only) tool call
    print("tool name:    ", call["function"]["name"])
    print("arguments:    ", call["function"]["arguments"], " <- type:", type(call["function"]["arguments"]))

    args = json.loads(call["function"][___])    # JSON text -> Python dictionary
    print("parsed:       ", args, " <- type:", type(args), "| query =", args["query"])


<details>
<summary><b>Click here for the solution</b></summary>

```python
resp = llm.chat(messages, model=MODEL, tools=TOOLS)
msg = resp["choices"][0]["message"]

print("finish_reason:", resp["choices"][0]["finish_reason"])
print("content:      ", msg["content"])

if not msg.get("tool_calls"):
    print("The model answered without a tool call this time. Run the cell again.")
else:
    call = msg["tool_calls"][0]               # the first (usually: only) tool call
    print("tool name:    ", call["function"]["name"])
    print("arguments:    ", call["function"]["arguments"], " <- type:", type(call["function"]["arguments"]))

    args = json.loads(call["function"]["arguments"])    # JSON text -> Python dictionary
    print("parsed:       ", args, " <- type:", type(args), "| query =", args["query"])
```

</details>


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `llm.chat(messages, model=MODEL, tools=TOOLS)` sends **one** request: the whole message list, the model name and the tool descriptions. The answer `resp` is a dictionary.
- `resp["choices"][0]["message"]` reads from the outside in: the list of choices (we always ask for one), its first entry, that entry's message. `msg` is the model's reply as a dictionary.
- `resp["choices"][0]["finish_reason"]` says why the model stopped writing: `"tool_calls"` (it wants a tool), `"stop"` (it finished) or `"length"` (it hit the output limit and was cut off).
- `msg["content"]` is the reply text: `None` when the model only requested a tool, or, with some models, text written alongside the tool request (see the question below the cell).
- `msg.get("tool_calls")` is the list of requested tool calls, or `None` when there are none. `.get` is like `[...]` but returns `None` instead of raising `KeyError` when the key is missing, which is why it is the right way to test for an optional key. A model is free to answer with text right away, so the `if` branch handles that case honestly.
- `call = msg["tool_calls"][0]` is the first requested call, a dictionary with an `id` and a `function` entry; `call["function"]["name"]` is the tool name the model chose.
- `call["function"]["arguments"]` is a **string** containing JSON text, such as `'{"query": "EU AI Act"}'`, not a dictionary. `type(...)` prints the type (`<class 'str'>`) to make this visible.
- `json.loads(...)` ("load string") turns that text into a Python dictionary; afterwards `args["query"]` works. If the model had written broken JSON, this line would raise an error, which is why part 3 wraps it in `try`.

</details>


> **Key point.** `call["function"]["arguments"]` is a JSON **string**, not a dictionary. `json.loads` turns it into one; if the model wrote broken JSON, this is where it fails.


❓ **Q:** Look at the `content` line in the output above. With this model you will often see a complete answer, sometimes even with citations like `[1]` or `[2]`, although `finish_reason` is `tool_calls` and no tool has run yet. Where does that text come from, how trustworthy is it, and what does the loop in Part F do with it?
<details><summary>Click for answer</summary>

The text comes from the model's training data alone. At this point the model has seen the system prompt, the task and the tool descriptions, nothing else; the search has not been executed, so there is no result it could cite. The citations are invented: the mini-web does not even number its pages. The API allows a message to carry `content` and `tool_calls` at the same time, and models differ in how disciplined they are. Well-trained tool users leave `content` empty (`None`) or write one short sentence such as "Let me look that up"; `qwen3-30b` often writes a full answer anyway.

The loop handles this correctly: it checks `tool_calls` first, executes the tool and appends the observation; the `content` of this round is never printed or returned as the answer. The real answer is the one the model gives in a later round, when it answers **without** a tool call, after it has seen the tool results. So the order "reason, act, observe, decide" still holds; the model has merely talked ahead of itself. The lesson for reading agent traces: a confident sentence is not evidence that a tool was used. Only a text that appears after the tool results can be grounded in them.

</details>


❓ **Q:** Why is the full message list sent with every request, even though most of it was already sent before?
<details><summary>Click for answer</summary>

Because the model is stateless: it keeps nothing between requests. Whatever it should know in this round (system prompt, task, its own earlier answers, all tool results) must be in this request. That is exactly why long runs get expensive: every round re-sends, and re-pays, everything before it.

</details>


❓ **Q:** The retries inside the model call and the step cap in the loop both limit repetition. What is the difference?
<details><summary>Click for answer</summary>

Retries handle **infrastructure** errors (no answer at all) and do not count as agent steps; they repeat the *same* request. The step cap limits **agent rounds**: each round is a new request with a longer message list. Both are bounded, so neither can run forever.

</details>


## Part E: Part 3 of the agent: executing one tool call

The model asked for `web_search(query="EU AI Act")`. Now the **runtime** acts: look up the function, parse the arguments, run it, and package the result as a message. Whatever goes wrong becomes an observation for the model, not a crash. Fill the gaps.


<div style="text-align: center;">
  <img src="images/fig07_run_tool_flow.png" width="960">
</div>


In [ ]:
def run_tool(call):
    '''Run one requested call; errors become observations.'''
    try:
        fn = FUNCS[___]                       # unknown tool -> KeyError
        args = json.loads(call["function"]["arguments"])  # invalid JSON -> JSONDecodeError
        result = fn(___)                      # bad arguments -> TypeError
    except Exception as e:                        # any error becomes text
        result = f"TOOL ERROR: {e!r}"
    return {"role": "tool", "tool_call_id": ___,
            "content": str(result)[:4000]}        # characters, not tokens


<details>
<summary><b>Click here for the solution</b></summary>

```python
def run_tool(call):
    '''Run one requested call; errors become observations.'''
    try:
        fn = FUNCS[call["function"]["name"]]                       # unknown tool -> KeyError
        args = json.loads(call["function"]["arguments"])  # invalid JSON -> JSONDecodeError
        result = fn(**args)                      # bad arguments -> TypeError
    except Exception as e:                        # any error becomes text
        result = f"TOOL ERROR: {e!r}"
    return {"role": "tool", "tool_call_id": call["id"],
            "content": str(result)[:4000]}        # characters, not tokens
```

</details>


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `def run_tool(call):` takes one tool call, the dictionary we looked at in Part D, and returns a tool message.
- `try:` starts a block in which an error does not crash the program; Python jumps to `except` instead.
- `FUNCS[call["function"]["name"]]` finds the registered function for the requested name, or raises `KeyError` when the name is unknown: there is no path from an invented name to real code.
- `json.loads(call["function"]["arguments"])` turns the argument text into a dictionary, or raises `JSONDecodeError` when the text is not valid JSON.
- `fn(**args)` calls the function with the dictionary **unpacked** as named arguments: `{"query": "EU AI Act"}` becomes `web_search(query="EU AI Act")`. That is why the keys must match the parameter names of the Python function; a wrong or missing name raises `TypeError`. An error inside the tool itself is caught here as well.
- `except Exception as e:` catches all of these. `Exception` is the common ancestor of ordinary errors, so one line covers every case. The f-string `f"TOOL ERROR: {e!r}"` puts the error into the text; `!r` shows it in its repr form, which includes the error type, for example `KeyError('search_web')`.
- The return value is a **tool message**: `"role": "tool"`, the `tool_call_id` copied from the call so that the provider can match result and request (exactly one tool message per call is required), and the result as text. `str(result)` makes sure it is a string, `[:4000]` keeps at most 4000 characters: it bounds one observation, not the whole request.

</details>


> **Python note.** `**` in front of a dictionary spreads its key-value pairs as named arguments into the call. `{e!r}` in an f-string shows the value with quotes and type, so an empty or odd name is visible.


**Test every path.** The cell below builds five tool calls by hand, in exactly the shape the model sends them, and runs each through `run_tool`. Predict each result before you run it. No model is involved here, so the result is the same every time.


In [ ]:
def make_tool_call(name, arguments):
    '''Build a tool call as the model would send it (arguments: a dictionary).'''
    return {"id": "call_test", "type": "function",
            "function": {"name": name, "arguments": json.dumps(arguments)}}   # dict -> JSON text

broken_call = {"id": "call_test", "type": "function",
               "function": {"name": "web_search", "arguments": '{"query": "penalties"'}}   # missing }

tests = [
    ("correct call",          make_tool_call("web_search", {"query": "penalties"})),
    ("invented tool name",    make_tool_call("search_web", {"query": "penalties"})),
    ("broken JSON",           broken_call),
    ("wrong argument name",   make_tool_call("web_search", {"q": "penalties"})),
    ("tool fails internally", make_tool_call("web_search", {"query": None})),
]
for label, call in tests:
    out = run_tool(call)
    print(f"{label:22} -> {out['content'][:88]}")


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `make_tool_call` is a small helper function for this test only. It takes a tool name and an argument dictionary and returns a dictionary of the same shape as a real tool call from Part D. `json.dumps` ("dump to string", the reverse of `json.loads`) turns the argument dictionary into JSON text, because that is what the model sends: `json.dumps({"query": "penalties"})` gives the string `'{"query": "penalties"}'`.
- `broken_call` is written out by hand because broken JSON cannot be produced from a dictionary: `json.dumps` always writes valid JSON. Its `arguments` string lacks the closing brace. (The outer `'` are Python's string quotes, not part of the JSON text.)
- `tests` is a list of pairs `(label, call)`. The fifth test passes `None` as the query, which is valid JSON and a valid argument name but makes `query.lower()` fail inside `web_search`.
- `for label, call in tests:` unpacks each pair. `run_tool(call)` returns the tool message; `out['content'][:88]` prints the first 88 characters of its text. `{label:22}` pads the label to 22 characters so that the arrows line up.
- Expected: the first line is a search result, the other four start with `TOOL ERROR:` and name a different error each. The loop never crashed, and every message has the same shape.
- The four error messages, read one by one. Each is the `repr` of the exception that `run_tool` caught: the error type, then the message in parentheses.
  - `KeyError('search_web')`: `FUNCS["search_web"]` failed because the dictionary has no such key. Python's `KeyError` simply names the key it could not find. The invented tool name never reached any code.
  - `JSONDecodeError("Expecting ',' delimiter: line 1 column 22 (char 21)")`: `json.loads` reached the end of the 21-character text `{"query": "penalties"` while the object was still open; it names the comma it would accept next, but the real cause is the missing `}`.
  - `TypeError("web_search() got an unexpected keyword argument 'q'")`: the JSON was fine and gave `{"q": "penalties"}`, but `fn(**args)` then called `web_search(q="penalties")`, and `web_search` has no parameter named `q`. The keys of the argument dictionary must match the parameter names of the Python function exactly.
  - `AttributeError("'NoneType' object has no attribute 'lower'")`: the call was valid, `web_search(query=None)` started running, and failed inside, at `query.lower()`: `None` is an object of type `NoneType`, and it has no method `lower`. This error comes from the tool's own code, not from `run_tool`, and is caught by the same `except`.

</details>


> **Key point.** Errors become observations. The loop keeps running, and the model reads the `TOOL ERROR` text in the next round and can correct its call.

> **Common mistake.** Thinking 4000 characters means 4000 tokens. Tokens are word pieces; 4000 characters of English are roughly 1000 tokens.


❓ **Q:** Why should `run_tool` return an error *message* instead of letting the exception crash the program?
<details><summary>Click for answer</summary>

A crash ends the whole run, including all work done so far, because of one bad call. As an observation, the error reaches the model, which can usually fix it (correct the name, fix the arguments, try another query). Agency shows most clearly in how a system handles failure.

</details>


❓ **Q:** The query `None` produced an `AttributeError`, not a `TypeError`. Where did it happen, and why does it matter that `except Exception` catches it too?
<details><summary>Click for answer</summary>

`None.lower()` fails inside `web_search`, so it is an error *of the tool*, not of the arguments' shape (the argument name was right and the JSON was valid). The catch-all `except Exception` keeps such failures from crashing the agent as well, while the message still says what went wrong. In a real agent the tool might be a search service that is down; the policy is the same.

</details>


## Part F: Part 4 of the agent: the loop

Now the four parts work together. Each round: **reason** (one request), append the answer (**state grows**), **decide** (tool call or not), **act** and **observe** for every tool call, then the next round.


<div style="text-align: center;">
  <img src="images/fig08_loop_flow.png" width="960">
</div>


Complete the loop. It is the lecture's code with one extra `print` per round, so that you can watch the run. Fill the gaps.


In [ ]:
MAX_STEPS = 8
messages = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": task}]

for step in range(___):                            # stop: step cap
    resp = llm.chat(___, model=MODEL, tools=TOOLS)    # REASON
    msg = resp["choices"][0]["message"]
    messages.append(___)                            # state grows here
    print(f"round {step}: finish_reason={resp['choices'][0]['finish_reason']}, "
          f"tool calls={len(msg.get('tool_calls') or [])}")

    if not msg.get(___):                            # no action requested
        if resp["choices"][0]["finish_reason"] == "length":   # truncated, not done
            print("Answer cut off at token limit."); break
        print(msg["content"]); break                    # completed (or refused)

    for call in msg["tool_calls"]:                      # each requested call
        observation = ___                           # ACT
        messages.append(observation)                    # OBSERVE
else:
    print("Step budget exhausted, no answer.")


<details>
<summary><b>Click here for the solution</b></summary>

```python
MAX_STEPS = 8
messages = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": task}]

for step in range(MAX_STEPS):                            # stop: step cap
    resp = llm.chat(messages, model=MODEL, tools=TOOLS)    # REASON
    msg = resp["choices"][0]["message"]
    messages.append(msg)                            # state grows here
    print(f"round {step}: finish_reason={resp['choices'][0]['finish_reason']}, "
          f"tool calls={len(msg.get('tool_calls') or [])}")

    if not msg.get("tool_calls"):                            # no action requested
        if resp["choices"][0]["finish_reason"] == "length":   # truncated, not done
            print("Answer cut off at token limit."); break
        print(msg["content"]); break                    # completed (or refused)

    for call in msg["tool_calls"]:                      # each requested call
        observation = run_tool(call)                           # ACT
        messages.append(observation)                    # OBSERVE
else:
    print("Step budget exhausted, no answer.")
```

</details>


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `MAX_STEPS = 8` is the step cap. `messages` is rebuilt with the two starting messages, so that the run starts clean (Part D already appended nothing, but a rerun of this cell must not continue an old conversation).
- `for step in range(MAX_STEPS):` runs the body at most 8 times, with `step` counting 0, 1, 2, ... This **step cap** needs no cooperation from the model: a counter is just a number, so unlike the model it cannot make mistakes. It bounds the number of rounds, not the work inside one.
- `llm.chat(messages, model=MODEL, tools=TOOLS)` is the REASON phase: the whole list so far goes to the model. `messages.append(msg)` adds the model's answer at once; from now on the model also reads its own earlier answers.
- The `print` line is our window into the run. `msg.get('tool_calls') or []` is the list of tool calls, or an empty list when there are none (`or []` replaces `None` by `[]` so that `len` works in both cases). Inside an f-string that is already written with double quotes, the keys are written with single quotes.
- `if not msg.get("tool_calls"):` is the DECIDE phase: no tool call means the model needs nothing more from the world. We read `finish_reason`: `"length"` means the answer hit the output limit and was cut off, so we report that; otherwise we print the answer. `break` leaves the `for` loop in both cases. *Completed* and *refused* look the same here; a real agent would check further.
- `for call in msg["tool_calls"]:` handles each requested call; the model may request several tools in one answer. `run_tool(call)` is the ACT phase, `messages.append(observation)` the OBSERVE phase: the tool message joins the list, and the model sees it in the next round.
- `else:` after a `for` loop runs only if the loop finished **without** `break`: here, the step budget was used up without a final answer. That is the second exit.

</details>


> **Common mistake.** Treating every answer without a tool call as success. Ending is not succeeding.


### The run, read from your own message list

The list `messages` now records the whole run. Print it: one line per message with its role and a short preview. Compare with the walkthrough's trace: typically two rounds, two model requests, one tool call, five messages. The cell is given.


In [ ]:
for i, m in enumerate(messages):
    if m.get("tool_calls"):
        preview = "tool call -> " + ", ".join(f"{c['function']['name']}({c['function']['arguments']})" for c in m["tool_calls"])
    else:
        preview = str(m["content"])[:90].replace("\n", " ")
    print(f"[{i}] {m['role']:9} {preview}")


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `enumerate(messages)` hands out pairs `(position, message)`, so `i` is the index of the message in the list and `m` the message dictionary.
- For an assistant message that requested tools, the preview lists the calls: the compact loop `for c in m["tool_calls"]` takes each call and calls it `c`, the f-string builds `name(arguments)`, and `", ".join(...)` glues several calls with commas.
- For every other message, `str(m["content"])[:90]` is the first 90 characters of the text (`str` because the content can be `None`), and `.replace("\n", " ")` removes line breaks so that each message stays on one line.
- `{m['role']:9}` pads the role to 9 characters so that the previews line up.

</details>


❓ **Q:** Put the final answer next to the tool message it cites: print `messages[3]["content"]`, or look at `PAGES`. Which parts of the answer come from the page, and which parts did the model add? Is the citation attached to the right page?
<details><summary>Click for answer</summary>

Only a few facts come from the page, typically the dates and the phrase "risk-based rules". Everything else, the adjectives, the aims of the Act, often the four risk levels, comes from the model's training data, even when it sits in the same sentence as a citation. A grounded answer is still generated text: the observation is one more input to the model, not a template it copies from. Two consequences. First, a citation marks where a sentence came from only as reliably as the model attached it; check whether the cited page actually contains the claim. Second, "grounded" is a property of individual claims, not of the answer as a whole. Later labs build checks for exactly this.

</details>


❓ **Q:** How many entries does the message list have at the end of a typical run, and where does each come from?
<details><summary>Click for answer</summary>

Five: `[0]` system prompt and `[1]` task (both written by us before the loop), `[2]` the model's tool call (round 0, appended right after the request), `[3]` the tool result (round 0, appended by `run_tool`), and `[4]` the model's final text answer (round 1). The step cap of 8 was never needed: the model stopped on its own. A model that searches twice produces seven entries; the structure is the same.

</details>


### `for ... else`: the second exit

`for ... else` is rare in Python and surprises many people. Read the `else` as *"else, if we never broke out"*. Fill the gap so that the `else` block is skipped.


<div style="text-align: center;">
  <img src="images/fig09_for_else.png" width="960">
</div>


In [ ]:
for k in range(4):
    print("round", k)
    if k == 1:                 # make the loop leave in round 1 ...
        ___
else:
    print("never broke out")    # ... so this line must NOT be printed


<details>
<summary><b>Click here for the solution</b></summary>

```python
for k in range(4):
    print("round", k)
    if k == 1:                 # make the loop leave in round 1 ...
        break
else:
    print("never broke out")    # ... so this line must NOT be printed
```

</details>


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `range(4)` produces 0, 1, 2, 3; without the `if`, the loop would print four rounds and then run the `else` block, because it finished without `break`.
- `break` ends the loop immediately in round 1. Because the loop was left through `break`, the `else` block is skipped: "never broke out" is not printed.
- In the agent, `break` is the "answer received" exit and the `else` block is the "step budget exhausted" exit. The two can never both run.

</details>


❓ **Q:** Why is the step cap essential even when the model usually stops after two rounds?
<details><summary>Click for answer</summary>

Because "usually" is not a guarantee. A model can keep calling tools (repeating a failed search, chasing a misleading result), and every round re-sends a longer list. The step cap is the one exit that does not depend on the model behaving well. A production agent would also record *which* stop condition fired.

</details>


### The whole agent in one piece: run it on a simple example

You have now seen every line. The cell below is the complete program once more, wrapped in a function `run_agent(task)` so that you can run it on any question with one line. It is the Part F loop without the diagnostic prints, returning the answer. The cell is given; run it, read the answer and the trajectory, then change the question and run it again (one or two requests per run).


In [ ]:
def run_agent(task, max_steps=8):
    '''The minimal agent as one function: returns (answer, messages).'''
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": task}]
    for step in range(max_steps):                          # stop: step cap
        resp = llm.chat(messages, model=MODEL, tools=TOOLS)   # REASON
        msg = resp["choices"][0]["message"]
        messages.append(msg)                               # state grows here
        if not msg.get("tool_calls"):                      # no action requested
            if resp["choices"][0]["finish_reason"] == "length":
                return None, messages                      # cut off
            return msg["content"], messages                # completed (or refused)
        for call in msg["tool_calls"]:
            messages.append(run_tool(call))                # ACT + OBSERVE
    return None, messages                                  # step budget exhausted

question = "How high can the fines under the EU AI Act be? One sentence, with the page id as source."
answer, transcript = run_agent(question)

print("ANSWER:", answer)
print("\nTRAJECTORY:")
for m in transcript:
    for call in (m.get("tool_calls") or []):
        print("  tool call:", call["function"]["name"], call["function"]["arguments"])
print(f"  {len(transcript)} messages in the final list")


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `def run_agent(task, max_steps=8):` wraps the loop in a function. `max_steps` replaces the global `MAX_STEPS`; its default value 8 is used when the caller does not pass one.
- The body is the Part F loop with two changes: `return` instead of `print` and `break`, and no `else` block. A `return` ends the function at once, so the three exits become three `return` lines: cut off (`None`), completed (the answer text), and, after the loop, step budget exhausted (`None` again). The line after the loop is reached only when the loop ran all its rounds without returning, exactly what `for ... else` expressed before.
- `messages.append(run_tool(call))` is act and observe in one line: run the tool and append its message.
- `answer, transcript = run_agent(question)` unpacks the two returned values. We call the message list `transcript` here, because it is the record of the whole run.
- The trajectory printout goes through the transcript and, for every message with tool calls, prints the tool name and arguments; `m.get("tool_calls") or []` yields an empty list for messages without any, so the inner loop simply does nothing for them.

</details>


## Part G: Break it on purpose (optional)

If time remains. No gaps, two small experiments with the real model:

1. **Rename the tool on one side only.** The cell below replaces `FUNCS` by a table with the key `"websearch"` and runs `run_agent` again. The model still uses the name from `TOOLS`, so every call ends in `TOOL ERROR: KeyError(...)`: the common mistake from Part C, seen from the inside. Watch whether the model gives up, keeps trying, or answers without a source. The last line restores `FUNCS`.
   If the output shows `tool calls: 0` and no `observation:` line, the model did not call the tool at all and answered from its training data, despite the system prompt. Then there is nothing for `run_tool` to fail on, and no error appears. Compare the numbers in the answer with `PAGES["penalties"]` (such an answer often quotes the figures of the 2021 draft, 30 million and 6 percent, and cites a source that does not exist), then run the cell again until a tool call appears. This is the Part D observation one step further: the same ungrounded text, now returned as the final answer, and the loop cannot tell.
2. **Tighten the cap.** Call `run_agent(question, max_steps=1)`. Which exit fires, and why is `None` the honest answer here?


In [ ]:
FUNCS = {"websearch": web_search}          # wrong on purpose: TOOLS still says "web_search"

answer, transcript = run_agent(question, max_steps=3)
n_calls = 0
for m in transcript:
    n_calls += len(m.get("tool_calls") or [])
    if m["role"] == "tool":
        print("observation:", m["content"][:80])
print("tool calls:", n_calls)
print("ANSWER:", str(answer)[:200])

FUNCS = {"web_search": web_search}         # restore the correct table


## Wrap-up: six facts about the minimal agent

| | |
|---|---|
| **The model never runs code.** | It only returns text and tool calls. The runtime executes. |
| **The model knows a tool only from its JSON description.** | Name, description, parameters. The description is prompt text. |
| **`FUNCS` is the only way from a name to a function.** | Not in the table, never executed. |
| **The whole message list is sent every round.** | That is why long runs get expensive. |
| **Every error becomes an observation.** | Unknown tool, bad arguments, failed call: the model reads a `TOOL ERROR` and can react. |
| **The loop has two exits.** | No tool call (classify it: completed or cut off) and the step cap (`for ... else`). |

### Python features used in these fifty lines

| Feature | Meaning |
|---|---|
| `import` | bring in code from a module or library |
| `def`, `return` | define a function; `return` hands a value back and ends the function |
| `{key: value}` | dictionary: a lookup table (messages, `TOOLS`, `FUNCS`) |
| `[a, b]` | list: an ordered collection (`messages` grows) |
| `d["key"]`, `lst[0]`, `d.get("key")` | look up a key, pick a position, look up a key that may be missing |
| `for x in range(n)` | repeat n times, x counts from 0 |
| `try` / `except` | run code; on an error jump to `except` instead of crashing |
| `fn(**args)` | call `fn` with the dictionary `args` spread as named arguments |
| `text[:4000]` | slice: the first 4000 characters |
| `f"...{x}..."` | f-string: put the value of `x` into the text |
| `for ... else` | `else` runs only if the loop ended without `break` |

**Next.** In the next lab you build this loop yourself around the course model, give it two tools over a small offline web, instrument it, and then break it on purpose: runaway loops, oversized pages, derailment.


❓ **Q:** In one sentence each: what do part 1, part 2, part 3 and part 4 of the agent do?
<details><summary>Click for answer</summary>

Part 1 sets up the client and the model name and declares the tool twice: as a Python function and as a JSON description, joined by the dispatch table `FUNCS`. Part 2 sends the whole message list to the model, with bounded retries. Part 3 executes one requested tool call safely and returns its result (or error) as a tool message. Part 4 is the loop: reason, append, decide, act and observe, until the model stops or the step cap ends the run.

</details>
